# 22. Ders: Hayalden Ekrana (Metinden Görüntü Üretimi / Text-to-Image)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Midjourney, DALL-E veya Stable Diffusion gibi yapay zekalar sizce internetten fotoğraf bulup onları birleştirerek mi (fotomontaj) resim yapıyor? Hayır!

Bu sistemler 'Diffusion' (Yayılım) adı verilen harika bir fizik kuralıyla çalışır. Bir bardak temiz suya bir damla mürekkep damlattığınızı hayal edin. Mürekkep yavaş yavaş suya yayılır ve suyu bulanıklaştırır. Yapay zeka eğitimi sırasında milyonlarca kedi fotoğrafı yavaş yavaş (adım adım) kumlanarak 'televizyon karıncalanmasına' (Noise) dönüştürülür. Model bu karıncalanmayı geri almayı öğrenir. Siz ona "Bana uzayda bir kedi çiz" dediğinizde, ekrana tamamen rastgele bir televizyon karıncalanması (Noise) koyar ve o pikselleri adım adım temizleyerek, sanki bir mermeri yontan heykeltraş gibi, içinden o kediyi çıkarır.

## Adım 2: Teknik Terimler ve Sözlük

- Text-to-Image: Sadece yazı yazarak (Örn: "Kırmızı bir elma") sıfırdan tamamen yeni bir dijital resim üretme teknolojisi.
- Diffusion Model (Yayılım Modeli): Rastgele gürültüden (noise) başlayıp, pikselleri adım adım temizleyerek anlamlı resimler üreten algoritma türü.
- Noise (Gürültü): Televizyon karıncalanmasına benzeyen rastgele pikseller bütünü. Tüm yapay zeka resimleri bu karıncalanmadan doğar.
- Prompt (İstem): Modele ne çizmesini istediğinizi anlatan sihirli sözcükler.
- Negative Prompt (Negatif İstem): Modelin çizmesini KESİNLİKLE İSTEMEDİĞİNİZ şeyler (Örn: "bulanık, çirkin, 6 parmak").

## Adım 3: Ortam Kurulumu (Diffusers Kütüphanesi)

Görüntü üreten modeller çok büyük ve ağırdır (Genellikle bir ekran kartına / GPU'ya ihtiyaç duyarlar). Hugging Face'in resim çizen ünlü kütüphanesi olan `diffusers`'ı kuracağız.

*Not: Bu dersi Google Colab'de çalıştırırken üst menüden 'Runtime (Çalışma Zamanı) -> Change runtime type -> Hardware Accelerator: T4 GPU' seçeneğinin aktif olduğundan emin olun.*

In [ ]:
# Terminal komutlarıyla gerekli çizim kütüphanelerini yüklüyoruz.
!pip install diffusers transformers accelerate

## Adım 4: Yapay Zeka Ressamını (Stable Diffusion) İçeri Aktarma

Dünyanın en çok bilinen açık kaynaklı resim modeli 'Stable Diffusion'ı (Kararlı Yayılım) koda davet ediyoruz.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch # Yapay zekanın GPU (Ekran kartı) üzerinde çalışmasını sağlayan motor

# Modeli indiriyoruz (Bu işlem internet hızınıza göre 2-3 dakika sürebilir, devasa bir ressam indiriyoruz!)
# float16: Modelin hafızada daha az yer kaplaması için kullandığımız bir sıkıştırma türüdür.
model_kimligi = "runwayml/stable-diffusion-v1-5"
ressam = StableDiffusionPipeline.from_pretrained(model_kimligi, torch_dtype=torch.float16)

# Ressamı bilgisayarın işlemcisinden (CPU) alıp Ekran Kartına (cuda/GPU) taşıyoruz ki hızlı çizsin.
ressam = ressam.to("cuda")

print("Ressam (Stable Diffusion) fırçasını aldı ve bekliyor!")

## Adım 5: Modele Görüntü Çizmesi İçin Prompt Hazırlama

Modeller (şu an için) İngilizce anlarlar. Ne kadar detaylı kelimeler (kamera açısı, ışık türü, tarz) girersek o kadar harika resimler çizerler.

In [ ]:
# Çok detaylı bir Prompt (İstem) yazıyoruz
hayalimdeki_resim = "A cute astronaut cat exploring a glowing neon forest, cinematic lighting, highly detailed, 4k resolution"

# Resimde ASLA olmamasını istediğimiz şeyler
istenmeyenler = "ugly, blurry, bad anatomy, deformed, dark"

print("Talebiniz ressama iletildi. Çizim başlıyor...")

## Adım 6: Modeli Çalıştırma ve Şaheseri Üretme

Aşağıdaki kodu çalıştırdığınızda, model bir karıncalanma başlatacak ve onu 50 adımda temizleyerek kediyi (veya sizin yazdığınız şeyi) çizecektir. Bu işlem GPU hızınıza göre 10-30 saniye sürer.

In [ ]:
# Ressamdan resmi çizmesini istiyoruz (.images[0] diyerek ilk resmi alıyoruz)
olusan_resim = ressam(prompt=hayalimdeki_resim, negative_prompt=istenmeyenler).images[0]

print("Çizim tamamlandı!")

## Adım 7: Çıkan Görüntüyü Ekranda Gösterme

Tıpkı 19. Derste yaptığımız gibi resmi Matplotlib ile ekrana asalım.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))
plt.imshow(olusan_resim)
plt.axis("off") # Kenardaki cetvelleri kapat
plt.title("Yapay Zeka Tarafından Çizilen Orijinal Eser")
plt.show()

## Adım 8: Yaratıcılık Seviyesi (Guidance Scale) Nedir?

Bazen model sizin yazdığınız kelimeleri pek takmaz, kafasına göre çizer. Sizin kelimelerinize ne kadar SIKI SIKIYA bağlı kalacağını belirten ayara `guidance_scale` denir.
- Değer 3 veya 4 ise: Model çok özgürdür, kafasına göre ilaveler yapar.
- Değer 7.5 ise: Standart, hem sizin dediğinizi yapar hem biraz yorum katar.
- Değer 12 ve üzeri ise: Birebir sadece yazdığınız kelimelere odaklanır, resim bazen çok yapay durabilir.

In [ ]:
# Örnek kullanım (Çalıştırmak isterseniz '#' işaretini kaldırın):
# resim_2 = ressam(prompt="A red sports car on the moon", guidance_scale=9.0).images[0]
# plt.imshow(resim_2)
# plt.show()

## Adım 9: Dene ve Keşfet - Kendi Hayal Dünyanı Çizdir

Adım 5'teki `hayalimdeki_resim` değişkeninin içini tamamen silin. İçerisine fantastik bir fikir yazın (Örn: "A flying pirate ship made of pizza, floating in space, cartoon style"). Kodu tekrar çalıştırıp arkadaşlarınızın veya ailenizin aklını başından alacak sanat eserleri üretin.

## Adım 10: Sıra Sende Görevi - Çizgi Roman Paneli (Comic Book) Üretici

Problem: Bir dijital çizgi roman yazıyorsunuz ve panelleri (kareleri) yapay zekaya çizdirmek istiyorsunuz. Yazdığınız her promptun otomatik olarak bir çizgi roman karesine (Comic book style) dönüşmesini sağlayan bir fonksiyon yazın.

Gereksinimler:
1. `cizgi_roman_ciz(sahne_aciklamasi)` adında bir def (fonksiyon) oluşturun.
2. Fonksiyonun içinde `f-string` kullanarak kullanıcının açıklamasını alın ve sonuna büyülü kelimeleri ekleyin: `f"{sahne_aciklamasi}, comic book style, graphic novel, ink drawing, highly detailed"`
3. Negatif prompt olarak `istenmeyenler = "photo, realistic, 3d render"` ekleyin (Çünkü çizgi roman istiyoruz, fotoğraf değil).
4. Ressam modelini çağırıp resmi çizdirin ve `plt.imshow` ile ekrana basın.
5. Dışarıda fonksiyonu `cizgi_roman_ciz("A superhero standing on a tall building in the rain")` diyerek test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 22. Dersi tamamlayarak dünyayı kasıp kavuran Üretken Yapay Zeka'nın (Generative AI) en sihirli alanlarından biri olan Text-to-Image (Metinden Görüntüye) teknolojisini kendi ellerinizle kodladınız.

Bir sonraki derste, piksellerin dünyasından çıkıp "Ses Dalgalarının" (Frekansların) dünyasına gireceğiz ve bizim konuşmalarımızı metne döken Siri/Alexa algoritmalarını (Speech Recognition) inceleyeceğiz.